# 05 常態分布與模擬（講義 06a / 06b）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

## 常態分布 ND(μ, σ)

⚠ scipy 的 `scale` 是**標準差** σ，不是變異數。

In [ ]:
mean, stdev = 80, 10
print(f"f(70) = {stats.norm.pdf(x=70, loc=mean, scale=stdev):0.4f}")
print(f"F(70) = {stats.norm.cdf(x=70, loc=mean, scale=stdev):0.4f}")
print(f"x for F(x) = 0.8: {stats.norm.ppf(q=0.8, loc=mean, scale=stdev):0.4f}")

## 直接算 vs 標準化（z）— 講義 06a 例題

In [ ]:
mean, stdev = 6.5, 2
print(f'Pr(X >= 10) = 1 - F(10) = {1 - stats.norm.cdf(x=10, loc=mean, scale=stdev):0.4f}')
print(f'Pr(X <= 0)  = F(0)      = {stats.norm.cdf(x=0, loc=mean, scale=stdev):0.4f}')
print(f'Using SND: Pr(Z >= {(10 - mean) / stdev}) = {1 - stats.norm.cdf(x=(10 - mean) / stdev):0.4f}')
print(f'x for Pr(X <= x) = 0.6: {stats.norm.ppf(q=0.6, loc=mean, scale=stdev):0.4f}')
print(f'               via z : {stats.norm.ppf(q=0.6) * stdev + mean:0.4f}')

In [ ]:
X = sk.Normal(6.5, 2)
X.P('X >= 10')
X.quantile(0.6)
X.quantile(0.05, tail='right')     # Pr(X ≥ x) = 0.05
sk.plot_dist(X, '4 <= X <= 8')
plt.show()

## 常用 z 值

In [ ]:
for q in [0.90, 0.95, 0.975, 0.99, 0.995]:
    print(f'z_{1 - q:.3f} = {stats.norm.ppf(q):.4f}')

## 二項的常態近似（連續校正）

$X \sim Bi(n, p) \approx ND(np, \sqrt{np(1-p)})$，$\Pr(a \le X \le b) \approx \Pr(a - 0.5 \le Y \le b + 0.5)$

In [ ]:
trial_cnt, prob_success = 50, 0.5
prob_bi = stats.binom.cdf(k=30, n=trial_cnt, p=prob_success) - stats.binom.cdf(k=24, n=trial_cnt, p=prob_success)
mean = trial_cnt * prob_success
stdev = (trial_cnt * prob_success * (1 - prob_success)) ** (1 / 2)
prob_without = stats.norm.cdf(x=30, loc=mean, scale=stdev) - stats.norm.cdf(x=25, loc=mean, scale=stdev)
prob_with = stats.norm.cdf(x=30.5, loc=mean, scale=stdev) - stats.norm.cdf(x=24.5, loc=mean, scale=stdev)
print(f'Pr(25 <= X <= 30)     = {prob_bi:0.4f}')
print(f'Pr(25 <= Y <= 30)     = {prob_without:0.4f}')
print(f'Pr(24.5 <= Y <= 30.5) = {prob_with:0.4f}')

In [ ]:
sk.normal_approx(50, 0.5, '25 <= X <= 30')
plt.show()

## 抽樣平均數的分布（中央極限定理）

$E[\bar{X}] = \mu$，$\operatorname{Var}(\bar{X}) = \sigma^2 / n$

In [ ]:
def simulate_sample_mean(mean, stdev, exp_cnt, sample_size):
    population = list(stats.norm.rvs(loc=mean, scale=stdev, size=1000, random_state=1))
    sample_means = [statistics.mean(random.sample(population, sample_size)) for _ in range(exp_cnt)]
    print(f'n = {sample_size}: mean of means {statistics.mean(sample_means):0.4f}, '
          f'variance {statistics.pvariance(sample_means):0.4f} (theory {stdev ** 2 / sample_size})')

random.seed(115)
for n in [4, 9, 16]:
    simulate_sample_mean(80, 4, 500, n)

In [ ]:
means = sk.sample_means(sk.Expon(1), n=30, reps=2000, seed=1)   # 母體不是常態也會趨近常態
plt.show()

## Bessel 校正：為什麼樣本變異數除以 n − 1（講義 06b）

In [ ]:
rng = np.random.default_rng(1)
samples = rng.normal(0, 2, size=(20000, 5))      # 母體變異數 = 4
print('除以 n    :', samples.var(axis=1, ddof=0).mean().round(3))
print('除以 n - 1:', samples.var(axis=1, ddof=1).mean().round(3))

## 蒙地卡羅：估 π、三門問題

In [ ]:
random.seed(115)
point_cnt = 2000
inside = sum(((random.uniform(0, 2) - 1) ** 2 + (random.uniform(0, 2) - 1) ** 2) <= 1 for _ in range(point_cnt))
print('pi ≈', 4 * inside / point_cnt)

sk.estimate_pi(2000, seed=115)
sk.monty_hall(10000, seed=115)

## 亂數小提醒
- 要讓結果可重現：`random.seed(...)`、`np.random.default_rng(seed)`、scipy 加 `random_state=...`
- `stats.norm.rvs(loc, scale, size)`、`random.sample(population, k)`（不放回）、`random.choices(population, k=...)`（放回）